# S2 Demo — K-Means Segmentation + PCA (Topic 2.2)
Online Retail II transactions → **RFM** per customer → choose K → cluster → project → **interpret**.

In [ ]:
import sys, pathlib
p = pathlib.Path.cwd()
while not (p / 'src').exists():
    p = p.parent
sys.path.insert(0, str(p / 'src')); REPO = p
import warnings; warnings.filterwarnings('ignore')

In [ ]:
from course_utils import data
tx = data.load_raw('online_retail')
rfm = data.make_rfm(tx)
print(rfm.shape)
rfm.describe().round(1)

## Standardize, then let elbow + silhouette argue about K

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt

Xs = StandardScaler().fit_transform(rfm[['recency_days', 'frequency', 'monetary']])
inertia, sil = [], []
for k in range(2, 9):
    km = KMeans(n_clusters=k, n_init=10, random_state=42).fit(Xs)
    inertia.append(km.inertia_)
    sil.append(silhouette_score(Xs, km.labels_))
fig, ax = plt.subplots(1, 2, figsize=(10, 3.2))
ax[0].plot(range(2, 9), inertia, marker='o'); ax[0].set_title('elbow (inertia)')
ax[1].plot(range(2, 9), sil, marker='o'); ax[1].set_title('silhouette')
for a in ax: a.set_xlabel('K')
plt.tight_layout()

Both point near **K=4** — and 4 segments is what marketing can act on. Guides, not oracles.

In [ ]:
km = KMeans(n_clusters=4, n_init=10, random_state=42).fit(Xs)
rfm['cluster'] = km.labels_
from course_utils import viz
viz.plot_clusters_2d(Xs, km.labels_)

## The table that means something — cluster profiles (this is the task input)

In [ ]:
profile = rfm.groupby('cluster').agg(
    size=('CustomerID', 'count'), recency_days=('recency_days', 'mean'),
    frequency=('frequency', 'mean'), monetary=('monetary', 'mean')).round(1)
print(profile)
data.save_processed(rfm, 'retail_segments')

*(SageMaker note: a built-in K-Means estimator exists for data that doesn't fit in memory — same concept, `sagemaker.KMeans(k=4, ...)`.)*